<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

## Company Contact & Social Media Extraction

## Project Overview
**Task**: Generate a clean list of unique companies from executive_final.csv to prepare for automated collection of Customer Service emails, Investor Relations emails, and official social media pages (Facebook, X/Twitter, BlueSky).

**Dataset**: executives_final.csv  

---

## What This Pipeline Does?
1. Load: Reads the full executive dataset containing repeated company rows.
2. Filter: Removes all columns unrelated to company identity.
3. Deduplicate: Keeps only one row per company, producing a clean unique list.

Prepare: Outputs a CSV ready for the next task (automated web search for company contact information).

---

## Libraries Used

- **Packages**: `pandas`, `matplotlib`, `re`

---


## Pipeline

### Step 1: Configure / Knowing the data
```
INPUT_FILE = "executive_final.csv"
FINAL_FILE = "executive_titles_summary.csv"
```
### Step 2: Normalization
```
- Standardize text format (`.title()`)
- Remove punctuation and digits
- Create `title_clean` column
```
### Step 3: Categorization
Map each cleaned title to a normalized executive category

### Step 4: Visualization
- Bar Chart: Executive titles  
- Pie Chart: Distribution by executive category
- Histogram: Distribution of the executive titles

## Outputs:
1. CVS File: executive_titles_category_list 
2. Bar chatr: bar_chart.png
3. Pie chart: category_pie.png
4. Histogram: histogram_frequencies.png

# SET-UP

In [1]:
#!pip install duckduckgo_search
#!pip install ddgs

In [2]:
# Libraries
import re
import time
import requests
import pandas as pd
import tldextract
from urllib.parse import urlparse
from ddgs import DDGS
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [3]:
#Print some values:
# Configuration
INPUT_FILE = "executive_final.csv"

df = pd.read_csv(INPUT_FILE)
# Display preview
df_sample = df.head()
df_sample

,company,filing_type,executive_name,executive_title,confidence,executive_title_clean
0,1 800 FLOWERS COM INC,8-K,Moose Munch,Coo,spacy,Chief Operating Officer
1,"10x Genomics, Inc.",8-K,James Wilbur,Director,spacy,Director
2,1606 CORP.,8-K,Gregory Lambrecht,Chief Executive Officer,low,Chief Executive Officer
3,"1895 Bancorp of Wisconsin, Inc. /MD/",8-K,David Ball,Chief Executive Officer,spacy,Chief Executive Officer
4,"1stdibs.com, Inc.",8-K,Everette Taylor,Director,spacy,Director


# Dataframe with just companies names

In [4]:
# Step 1: Clean al the data
companies = df[['company']].drop_duplicates()
companies = companies.sort_values(by='company').reset_index(drop=True)
companies.to_csv(OUTPUT_FILE, index=False)
companies.head()

,company
0,1 800 FLOWERS COM INC
1,"10x Genomics, Inc."
2,1606 CORP.
3,"1895 Bancorp of Wisconsin, Inc. /MD/"
4,"1stdibs.com, Inc."


### Normalization of companies

In [5]:
# Normalization of companies

# Terms that are not acronyms
LEGAL_SUFFIXES = {
    "INC", "CORP", "CO", "LTD", "LLC", "PLC",
    "GROUP", "HOLDINGS", "HLDGS", "COM", "COMPANY"
}

# Simple domain pattern:
DOMAIN_RE = re.compile(
    r"^[A-Za-z0-9\-]+\.(com|org|net|io|co|ai|gov|edu|biz|info)$",
    flags=re.IGNORECASE
)

def is_acronym(word: str) -> bool:
    """
    True acronym:
    - 2–4 uppercase letters
    - letters only
    - not a legal suffix
    """
    return (
        word.isupper()
        and word.isalpha()
        and 2 <= len(word) <= 4
        and word not in LEGAL_SUFFIXES
    )

def is_mix_acronym(word: str) -> bool:
    """
    Mixed alphanumeric acronym like 3D, 5G, 2U.
    """
    return bool(re.match(r"^\d[A-Z]$", word))

def process_simple(part: str) -> str:
    """
    Normalize a single non-hyphen, non-domain token.
    """
    if not part:
        return part

    # Patterns like 8X8 to 8x8
    if re.match(r"^\d+[A-Za-z]\d+$", part):
        return part.lower()

    # True acronym (ETF, AAON, AB, FCP)
    if is_acronym(part):
        return part  # keep uppercase

    # Mixed acronym like 3D
    if is_mix_acronym(part):
        return part

    upper = part.upper()

    # Legal suffixes (Inc, Corp, Co, Ltd, LLC, PLC, etc.)
    if upper in LEGAL_SUFFIXES:
        if upper in {"LLC", "PLC"}:
            return upper  # keep uppercase for these
        else:
            return upper.capitalize() 

    # All caps but not acronym or legal suffix
    if part.isupper():
        return part.title()

    # Leading digits then letters
    if re.match(r"^\d+[A-Za-z]+$", part):
        digits = re.findall(r"^\d+", part)[0]
        letters = part[len(digits):]
        return digits + letters.capitalize()
    return part.capitalize()


def clean_token(word: str) -> str:
    """
    Token-level cleaning:
    - domain detection
    - dot cleanup for initials/suffixes
    - hyphen handling
    """
    if not word:
        return word

    # Domains
    if DOMAIN_RE.match(word):
        return word.lower()

    w = word

    # Remove dots in initials
    w = re.sub(r"\.(?=[A-Z]|$)", "", w)

    # Hyphenated words
    if "-" in w:
        parts = w.split("-")
        processed = [process_simple(p) for p in parts if p]
        return "-".join(processed)

    # Non-hyphen simple token
    return process_simple(w)


def normalize_company(name) -> str:
    """
    Full-company normalization:
    - remove SEC state codes (/DE/, /MD/, etc.)
    - remove commas
    - strip unwanted characters
    - collapse whitespace
    - normalize each token
    """
    if not isinstance(name, str):
        name = str(name)

    # Remove SEC state codes like
    name = re.sub(r"/[A-Za-z]{2}/", " ", name)

    # Remove commas
    name = name.replace(",", " ")

    # Remove unwanted characters except letters, digits, space, dot, ampersand, hyphen, slash
    name = re.sub(r"[^A-Za-z0-9\s.&\-/]", " ", name)

    # Collapse multiple spaces
    name = re.sub(r"\s+", " ", name).strip()

    if not name:
        return name

    tokens = name.split()
    cleaned_tokens = [clean_token(tok) for tok in tokens]

    return " ".join(cleaned_tokens)


# Apply to the deduplicated 'companies' dataframe
companies["company_clean"] = companies["company"].apply(normalize_company)

companies.head(30)

,company,company_clean
0,1 800 FLOWERS COM INC,1 800 Flowers Com Inc
1,"10x Genomics, Inc.",10X Genomics Inc
2,1606 CORP.,1606 Corp
3,"1895 Bancorp of Wisconsin, Inc. /MD/",1895 Bancorp Of Wisconsin Inc
4,"1stdibs.com, Inc.",1stdibs.com Inc
5,21Shares Core Ethereum ETF,21Shares Core Ethereum ETF
6,"22nd Century Group, Inc.",22Nd Century Group Inc
7,"2seventy bio, Inc.",2Seventy Bio Inc
8,374Water Inc.,374Water Inc
9,3D SYSTEMS CORP,3D Systems Corp


# Web Extraction

In [6]:
#SET UP: # Email regex
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# HTTP headers
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Noise / non-official domains
NOISE_DOMAINS = [
    "wikipedia.org",
    "yahoo.com",
    "news.yahoo.com",
    "bloomberg.com",
    "reuters.com",
    "marketwatch.com",
    "techcrunch.com",
    "crunchbase.com",
    "linkedin.com",
    "dwinnex.com",
    "wordpress.com",
    "blogspot.com",
    "medium.com",
    "zoominfo.com",
    "herokuapp.com",
    "sourceforge.net",
    "sec.gov",
    "bsky.app",        # for domain discovery only
    "facebook.com",
    "twitter.com",
    "x.com"
]

# simple in-memory caches to avoid repeated lookups
ddg_cache = {}
domain_cache = {}
html_cache = {}

In [7]:
#HELPER FUNCTIONS:

def ddg_urls(query, max_results=8):
    """
    Cached DuckDuckGo wrapper that returns a list of URLs.
    """
    if query in ddg_cache:
        return ddg_cache[query]

    urls = []
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)
            for r in results:
                url = r.get("href", "")
                if url:
                    urls.append(url)
    except Exception:
        pass

    ddg_cache[query] = urls
    return urls


def is_noise(url):
    return any(nd in url.lower() for nd in NOISE_DOMAINS)

def extract_domain(url):
    if not url:
        return None
    try:
        ext = tldextract.extract(url)
        if not ext.domain or not ext.suffix:
            return None
        return f"{ext.domain}.{ext.suffix}".lower()
    except Exception:
        return None


def looks_like_company_domain(domain, company):
    """
    Score how much the domain resembles the company name.
    We use simple token overlap; more overlap = better.
    """
    if not domain or not company:
        return 0

    company_words = [
        w.lower()
        for w in re.split(r"\W+", company)
        if len(w) > 2 and not w.isdigit()
    ]
    if not company_words:
        return 0

    score = 0
    for w in company_words:
        if w in domain:
            score += 1
    return score


def discover_domain(company):
    """
    Return the best-guess corporate domain for a company,
    or None if we can't find anything plausible.
    """
    if company in domain_cache:
        return domain_cache[company]

    queries = [
        f"{company} official website",
        f"{company} corporate website",
        f"{company} company site"
    ]

    best_domain = None
    best_score = 0

    for q in queries:
        urls = ddg_urls(q, max_results=10)

        for u in urls:
            if is_noise(u):
                continue

            dom = extract_domain(u)
            if not dom:
                continue

            score = looks_like_company_domain(dom, company)
            if score > best_score:
                best_score = score
                best_domain = dom

    # minimal threshold: at least 1 token match
    if best_score == 0:
        best_domain = None

    domain_cache[company] = best_domain
    return best_domain

def fetch_html(url):
    """
    Fetch HTML with basic retry and caching.
    """
    if not url:
        return None

    if url in html_cache:
        return html_cache[url]

    for attempt in range(2):
        try:
            r = requests.get(url, headers=HEADERS, timeout=8)
            if r.status_code == 200:
                html_cache[url] = r.text
                return r.text
        except Exception:
            time.sleep(1)

    html_cache[url] = None
    return None


def extract_emails_from_html(html):
    if not html:
        return []
    emails = re.findall(EMAIL_RE, html)
    return list(set(emails))
    
def find_page_on_domain(company, domain, keyword):
    """
    Use DuckDuckGo to find a page on the given domain for a keyword
    (e.g., 'investor relations', 'contact', 'support').
    """
    if not domain:
        return None

    query = f"{company} {keyword} site:{domain}"
    urls = ddg_urls(query, max_results=10)

    for u in urls:
        if domain in u.lower() and not is_noise(u):
            return u
    return None

In [8]:
#EXTRACTORS:

#EMAIL:
def get_investor_info(company, domain):
    """
    Try to find an IR page and any email on it.
    Fallback to generic 'investor' or 'ir' keywords.
    """
    if not domain:
        return None, None

    # primary try
    page = find_page_on_domain(company, domain, "investor relations")
    if not page:
        # fallback keywords
        for kw in ["investor", "ir"]:
            page = find_page_on_domain(company, domain, kw)
            if page:
                break

    if not page:
        return None, None

    html = fetch_html(page)
    emails = extract_emails_from_html(html)

    # prefer emails that look like IR-related
    if emails:
        for e in emails:
            if any(tag in e.lower() for tag in ["ir@", "investor@", "investors@", "shareholder"]):
                return e, page
        return emails[0], page

    return None, page

#Customer Service:
def get_customer_service_info(company, domain):
    """
    Try to find a customer service / support / contact email.
    """
    if not domain:
        return None, None

    # primary keyword
    page = find_page_on_domain(company, domain, "customer service")
    if not page:
        # fallback keywords
        for kw in ["contact", "support", "help"]:
            page = find_page_on_domain(company, domain, kw)
            if page:
                break

    if not page:
        return None, None

    html = fetch_html(page)
    emails = extract_emails_from_html(html)

    # prefer 'support', 'service', 'help', 'info'
    if emails:
        for e in emails:
            if any(tag in e.lower() for tag in ["support@", "service@", "help@", "info@", "care@"]):
                return e, page
        return emails[0], page

    return None, page


#FACEBOOK:
def find_facebook(company):
    query = f"{company} official facebook page"
    urls = ddg_urls(query, max_results=8)
    for u in urls:
        ul = u.lower()
        if "facebook.com/" in ul and "php" not in ul and "sharer" not in ul:
            return u
    return None

# TWITTER
def find_twitter(company):
    query = f"{company} official twitter x"
    urls = ddg_urls(query, max_results=8)
    for u in urls:
        ul = u.lower()
        if ("twitter.com/" in ul or "x.com/" in ul):
            if all(bad not in ul for bad in ["/status/", "/intent/", "/hashtag/", "/search"]):
                return u
    return None

#BLUSKY
def find_bluesky(company):
    query = f"{company} official bluesky"
    urls = ddg_urls(query, max_results=8)
    for u in urls:
        ul = u.lower()
        if "bsky.app/profile/" in ul:
            return u
    return None

In [9]:
#EXTRACTOR LOOP:
def extract_for_company(row):
    """
    Given a row from 'companies', return a dict with extracted info.
    """
    company_raw = row["company"]
    company_clean = row["company_clean"]

    name_for_search = company_clean  # could also lowercase if you want

    # 1. Discover domain
    domain = discover_domain(name_for_search)

    ir_email, ir_page = None, None
    cs_email, cs_page = None, None

    if domain:
        ir_email, ir_page = get_investor_info(name_for_search, domain)
        cs_email, cs_page = get_customer_service_info(name_for_search, domain)

    fb = find_facebook(name_for_search)
    tw = find_twitter(name_for_search)
    bs = find_bluesky(name_for_search)

    return {
        "company": company_raw,
        "company_clean": company_clean,
        "domain": domain,
        "ir_page": ir_page,
        "ir_email": ir_email,
        "cs_page": cs_page,
        "cs_email": cs_email,
        "facebook": fb,
        "twitter": tw,
        "bluesky": bs
    }

In [10]:
def run_production_extraction(companies,
                              out_file="company_contacts_full.csv",
                              save_every=50,
                              max_companies=None):
    """
    Run full extraction over 'companies' DataFrame.
    - save_every: save partial results every N companies
    - max_companies: limit to first N companies (for testing); None = all
    """
    results = []
    start_time = time.time()

    if max_companies is not None:
        df_iter = companies.iloc[:max_companies].reset_index(drop=True)
    else:
        df_iter = companies.reset_index(drop=True)

    for idx, row in tqdm(df_iter.iterrows(), total=len(df_iter), desc="Extracting"):
        try:
            info = extract_for_company(row)
        except Exception as e:
            # don't crash the whole run; just log a blank row with error
            info = {
                "company": row["company"],
                "company_clean": row["company_clean"],
                "domain": None,
                "ir_page": None,
                "ir_email": None,
                "cs_page": None,
                "cs_email": None,
                "facebook": None,
                "twitter": None,
                "bluesky": None,
                "error": str(e)
            }

        results.append(info)

        # periodic checkpoint
        if (idx + 1) % save_every == 0:
            tmp_df = pd.DataFrame(results)
            tmp_df.to_csv(out_file, index=False)
            print(f"Checkpoint saved at {idx + 1} companies → {out_file}")

        # gentle rate limiting
        time.sleep(1)

    final_df = pd.DataFrame(results)
    final_df.to_csv(out_file, index=False)

    elapsed = time.time() - start_time
    print(f"Completed {len(df_iter)} companies in {elapsed/60:.1f} minutes.")
    print(f"Final results saved to: {out_file}")

    return final_df

# Set up for full extraction

In [11]:
save_every = 50        # save every 50 rows → ~90 checkpoints
rate_limit = 1         # sleep 1 second per company
max_companies = None   # None → run all rows
output_file = "company_contacts_full.csv"

Extracting:   0%|          | 0/10 [00:00<?, ?it/s]

Checkpoint saved at 5 companies → company_contacts_test10.csv
Checkpoint saved at 10 companies → company_contacts_test10.csv
Completed 10 companies in 4.1 minutes.
Final results saved to: company_contacts_test10.csv


,company,company_clean,domain,ir_page,ir_email,cs_page,cs_email,facebook,twitter,bluesky
0,1 800 FLOWERS COM INC,1 800 Flowers Com Inc,1800flowersinc.com,https://www.1800flowersinc.com/investors,None,https://www.1800flowersinc.com/,None,None,https://twitter.com/1800flowers,None
1,"10x Genomics, Inc.",10X Genomics Inc,10xgenomics.com,https://www.10xgenomics.com/legal/terms-of-use,support@10xgenomics.com,https://www.10xgenomics.com/support,support@10xgenomics.com,None,None,https://bsky.app/profile/did:plc:tdn46iumucy4s...
2,1606 CORP.,1606 Corp,None,None,None,None,None,https://www.facebook.com/RedChipCompanies/post...,https://twitter.com/1606corp,None
3,"1895 Bancorp of Wisconsin, Inc. /MD/",1895 Bancorp Of Wisconsin Inc,1895bancorpofwisconsin.com,https://1895bancorpofwisconsin.com/,None,https://1895bancorpofwisconsin.com/financials/...,None,https://www.facebook.com/reel/3076816265807466/,None,None
4,"1stdibs.com, Inc.",1stdibs.com Inc,1stdibs.com,https://www.1stdibs.com/about/careers/manager-...,None,https://www.1stdibs.com/about/careers/customer...,None,None,https://twitter.com/1stDibs,None
5,21Shares Core Ethereum ETF,21Shares Core Ethereum ETF,21shares.com,https://www.21shares.com/en-us/products-us/ceth,None,https://www.21shares.com/en-us/contact-us,info@21shares.com,None,None,None
6,"22nd Century Group, Inc.",22Nd Century Group Inc,xxiicentury.com,https://ir.xxiicentury.com/,mkreps@xxiicentury.com,https://xxiicentury.com/,None,None,None,None
7,"2seventy bio, Inc.",2Seventy Bio Inc,2seventybio.com,https://ir.2seventybio.com/news-releases/news-...,None,https://ir.2seventybio.com/sec-filings/sec-fil...,None,None,https://twitter.com/hashtag/2seventy?src=hasht...,None
8,374Water Inc.,374Water Inc,374water.com,https://374water.com/investor-news/,None,https://374water.com/,None,None,None,None
9,3D SYSTEMS CORP,3D Systems Corp,3dsystems.com,https://www.3dsystems.com/3d-printers,press@3dsystems.com,https://support.3dsystems.com/s/article/Contac...,bg-info@2x.png,None,None,None


In [ ]:
#SAVE TO CSV
full_df = run_production_extraction(
    companies,
    out_file="company_contacts_full.csv",
    save_every=50,
    max_companies=None
)

full_df